# 🚀 ULM-1.7B QLoRA SFT (Google Colab Training)

이 노트북은 **Qwen3-1.7B** 모델을 울산 방언 데이터셋(Ulsan Core, 53만 문장)으로 파인튜닝하는 Google Colab 전용 실행 노트북입니다.

### 📌 사전 준비
1. 상단 메뉴: **[런타임] ➔ [런타임 유형 변경] ➔ T4 GPU (무료)** 선택
2. 왼쪽 메뉴 **📁 파일(Files)** 패널에 로컬에서 압축한 `ulsan_dataset.zip` (28MB) 파일을 업로드하세요.

In [ ]:
# 1. 저장소 클론 및 작업 디렉터리 이동 (Colab 필수)
import os

if not os.path.exists("/content/ULM-1.7B"):
    !git clone https://github.com/UlsanLM-LAB/ULM-1.7B.git /content/ULM-1.7B

%cd /content/ULM-1.7B
!git pull origin main

In [ ]:
# 2. GPU 확인 및 필수 패키지 설치
!nvidia-smi

!pip install -q torch transformers datasets peft trl bitsandbytes accelerate
!pip install -q -e .

In [ ]:
# 3. ulsan_dataset.zip 자동 압축 해제 및 확인
import os
import shutil
import zipfile

target_dir = "/content/ULM-1.7B/data/private/ulsan_dataset"
os.makedirs(target_dir, exist_ok=True)

zip_candidates = [
    "/content/ulsan_dataset.zip",
    "/content/ULM-1.7B/ulsan_dataset.zip",
    "ulsan_dataset.zip",
]
found_zip = next((p for p in zip_candidates if os.path.exists(p)), None)

if found_zip:
    print(f"📦 압축 파일 발견: {found_zip}")
    with zipfile.ZipFile(found_zip, "r") as z:
        for member in z.namelist():
            fname = os.path.basename(member)
            if fname.endswith(".jsonl") or fname.endswith(".json"):
                with z.open(member) as src, open(os.path.join(target_dir, fname), "wb") as dst:
                    shutil.copyfileobj(src, dst)
    print(f"✓ 데이터셋 배치 완료: {target_dir}")
    !ls -lh {target_dir}
elif os.path.exists(os.path.join(target_dir, "train.jsonl")):
    print(f"✓ 이미 데이터셋 파일이 존재합니다: {target_dir}")
    !ls -lh {target_dir}
else:
    raise FileNotFoundError(
        "❌ 'ulsan_dataset.zip' 파일이 없습니다! Colab 왼쪽 파일 패널(/content/)에 업로드해주세요."
    )

In [ ]:
# 4. ULM-1.7B QLoRA SFT 학습 실행
# (체크포인트는 outputs/qwen3-1.7b-sft에 주기적으로 자동 저장됩니다)
!python scripts/train_sft.py --config configs/sft/qwen3_1.7b_qlora.yaml

In [ ]:
# 5. 학습 완료된 어댑터로 울산 방언 추론 테스트
!python scripts/infer.py \
    --base-model Qwen/Qwen3-1.7B \
    --adapter-path outputs/qwen3-1.7b-sft \
    --prompt "오늘 날씨 참 좋다, 저녁에 밥 뭐 먹으러 갈래?" \
    --dialect-strength 2